# Create the Vizcarra 2023 Dataset for Classification
The Vizcarra 2023 dataset is designed for object detection. We need to work on making this a classification style dataset.

In [ ]:
import re

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw
from shapely.affinity import translate
from shapely.geometry import GeometryCollection, MultiPolygon, Polygon, box

try:
    import geopandas as gpd
except ModuleNotFoundError:
    gpd = None

try:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
except ModuleNotFoundError:
    plt = None
    Rectangle = None

try:
    from ipywidgets import IntSlider, interact
except ModuleNotFoundError:
    IntSlider = None
    interact = None

In [ ]:
dataset_dir = "/path/to/vizcarra-2023"
dataset_path = Path(dataset_dir)

# Metadata.
metadata_df = pd.read_csv(dataset_path / "rois.csv")
metadata_df.head()

In [ ]:
test_df = metadata_df[metadata_df["type"] == "test"].copy()
train_val_df = metadata_df[metadata_df["type"] != "test"].copy()

## WSI-Level Split

In [ ]:
def split_train_val_by_wsi(
    df,
    *,
    train_fraction=0.8,
    wsi_col="wsi_name",
    seed=2023,
    train_name="train",
    val_name="validation",
):
    """Assign a reproducible 80:20 split without leaking WSIs across splits."""
    split_df = df.copy()
    wsi_names = pd.Series(split_df[wsi_col].dropna().unique()).sample(
        frac=1,
        random_state=seed,
    )
    n_train = int(round(len(wsi_names) * train_fraction))
    train_wsis = set(wsi_names.iloc[:n_train])

    split_df["split"] = np.where(
        split_df[wsi_col].isin(train_wsis),
        train_name,
        val_name,
    )
    return split_df


train_val_split_df = split_train_val_by_wsi(train_val_df)
test_split_df = test_df.assign(split="hold-out")

train_val_split_df["split"].value_counts()

## ROI Tiling Helper

In [ ]:
CLASS_NAMES = ["pre_nft", "inft"]
CLASS_COLORS = {0: "#1f77b4", 1: "#d62728"}


def _safe_path_part(value):
    value = str(value or "unknown")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("_") or "unknown"


def _read_numeric_values(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return []

    values = []
    for line in path.read_text().splitlines():
        line = line.strip().replace(",", " ")
        if not line:
            continue
        values.extend(float(part) for part in line.split())
    return values


def read_boundary_polygon(boundary_path, image_size):
    """Read the ROI polygon from normalized or absolute corner coordinates."""
    width, height = image_size
    image_bounds = box(0, 0, width, height)
    values = _read_numeric_values(boundary_path)

    if len(values) < 6:
        return image_bounds
    if len(values) % 2 != 0:
        raise ValueError(f"Boundary file has an odd number of coordinates: {boundary_path}")

    coords = list(zip(values[::2], values[1::2]))
    if max(abs(value) for value in values) <= 1.0:
        coords = [(x * width, y * height) for x, y in coords]

    polygon = Polygon(coords)
    if not polygon.is_valid:
        polygon = polygon.buffer(0)

    polygon = polygon.intersection(image_bounds)
    return polygon if not polygon.is_empty else image_bounds


def _box_from_label_values(values, image_size, label_format="auto"):
    width, height = image_size
    cls = int(values[0])
    x1_or_cx, y1_or_cy, x2_or_w, y2_or_h = values[1:5]
    valid_formats = {"auto", "normalized_yolo", "absolute_yolo", "absolute_xyxy"}
    if label_format not in valid_formats:
        raise ValueError(f"label_format must be one of {sorted(valid_formats)}")

    is_normalized = all(0 <= value <= 1 for value in (x1_or_cx, y1_or_cy, x2_or_w, y2_or_h))
    if label_format == "normalized_yolo" or (label_format == "auto" and is_normalized):
        cx = x1_or_cx * width
        cy = y1_or_cy * height
        bw = x2_or_w * width
        bh = y2_or_h * height
        x1 = cx - bw / 2
        y1 = cy - bh / 2
        x2 = cx + bw / 2
        y2 = cy + bh / 2
        encoding = "normalized_yolo"
    elif label_format == "absolute_xyxy" or (
        label_format == "auto" and x2_or_w > x1_or_cx and y2_or_h > y1_or_cy
    ):
        x1, y1, x2, y2 = x1_or_cx, y1_or_cy, x2_or_w, y2_or_h
        encoding = "absolute_xyxy"
    else:
        cx, cy, bw, bh = x1_or_cx, y1_or_cy, x2_or_w, y2_or_h
        x1 = cx - bw / 2
        y1 = cy - bh / 2
        x2 = cx + bw / 2
        y2 = cy + bh / 2
        encoding = "absolute_yolo"

    geom = box(x1, y1, x2, y2).intersection(box(0, 0, width, height))
    return cls, geom, encoding


def read_label_boxes(label_path, image_size, *, label_format="auto"):
    """Read ROI labels as a GeoDataFrame in source-image pixel coordinates."""
    label_path = Path(label_path) if label_path is not None else None
    rows = []

    if label_path is not None and label_path.exists() and label_path.stat().st_size > 0:
        for line_number, line in enumerate(label_path.read_text().splitlines(), start=1):
            line = line.strip().replace(",", " ")
            if not line:
                continue

            parts = [float(part) for part in line.split()]
            if len(parts) < 5:
                raise ValueError(f"Expected 5 label values on {label_path}:{line_number}")

            cls, geom, encoding = _box_from_label_values(parts[:5], image_size, label_format)
            if cls not in (0, 1) or geom.is_empty or geom.area == 0:
                continue

            rows.append(
                {
                    "class_id": cls,
                    "encoding": encoding,
                    "object_area": geom.area,
                    "geometry": geom,
                }
            )

    columns = ["class_id", "encoding", "object_area", "geometry"]
    if gpd is None:
        return pd.DataFrame(rows, columns=columns)
    return gpd.GeoDataFrame(rows, columns=columns, geometry="geometry", crs=None)


def _iter_geometries(geom):
    if geom.is_empty:
        return
    if isinstance(geom, Polygon):
        yield geom
    elif isinstance(geom, (MultiPolygon, GeometryCollection)):
        for part in geom.geoms:
            yield from _iter_geometries(part)


def _polygon_mask_for_tile(roi_polygon, tile_geom, tile_size):
    local_roi = translate(
        roi_polygon.intersection(tile_geom),
        xoff=-tile_geom.bounds[0],
        yoff=-tile_geom.bounds[1],
    )
    mask = Image.new("L", (tile_size, tile_size), 0)
    draw = ImageDraw.Draw(mask)

    for polygon in _iter_geometries(local_roi):
        coords = [(float(x), float(y)) for x, y in polygon.exterior.coords]
        if len(coords) >= 3:
            draw.polygon(coords, fill=255)

    return mask


def _crop_and_mask_tile(image, roi_polygon, tile_geom, source_tile_size, outside_value):
    x1, y1, x2, y2 = (int(round(value)) for value in tile_geom.bounds)
    tile = Image.new(image.mode, (source_tile_size, source_tile_size), outside_value)

    crop_box = (
        max(0, x1),
        max(0, y1),
        min(image.width, x2),
        min(image.height, y2),
    )
    if crop_box[2] > crop_box[0] and crop_box[3] > crop_box[1]:
        tile.paste(image.crop(crop_box), (crop_box[0] - x1, crop_box[1] - y1))

    mask = _polygon_mask_for_tile(roi_polygon, tile_geom, source_tile_size)
    gray = Image.new(image.mode, (source_tile_size, source_tile_size), outside_value)
    return Image.composite(tile, gray, mask)


def _tile_box_records(label_boxes, tile_geom, source_tile_size, output_tile_size, threshold):
    if label_boxes.empty:
        return [], [0, 0]

    scale = output_tile_size / source_tile_size
    rows = []
    labels = [0, 0]
    x0, y0, _, _ = tile_geom.bounds

    intersections = label_boxes.copy()
    geometry = intersections["geometry"]
    if hasattr(geometry, "intersection"):
        intersections["intersection"] = geometry.intersection(tile_geom)
    else:
        intersections["intersection"] = geometry.map(lambda geom: geom.intersection(tile_geom))
    non_empty = intersections["intersection"].map(lambda geom: not geom.is_empty)
    intersections = intersections[non_empty].copy()

    for row in intersections.itertuples():
        fraction = row.intersection.area / row.object_area if row.object_area else 0.0
        included = fraction >= threshold
        if included:
            labels[int(row.class_id)] = 1

        bx1, by1, bx2, by2 = row.intersection.bounds
        rows.append(
            {
                "class_id": int(row.class_id),
                "class_name": CLASS_NAMES[int(row.class_id)],
                "fraction": float(fraction),
                "included": bool(included),
                "x1": (bx1 - x0) * scale,
                "y1": (by1 - y0) * scale,
                "x2": (bx2 - x0) * scale,
                "y2": (by2 - y0) * scale,
            }
        )

    return rows, labels


def tile_roi(
    roi_path,
    label_path,
    boundary_path,
    *,
    tile_size=512,
    source_magnification=40,
    target_magnification=20,
    object_fraction_threshold=0.5,
    min_roi_fraction=0.0,
    output_root=None,
    split=None,
    wsi_name=None,
    case_id=None,
    image_format="png",
    label_format="auto",
    include_partial_tiles=True,
    outside_value=(192, 192, 192),
    return_images=False,
):
    """Tile one ROI into non-overlapping classifier tiles and multilabel targets.

    `tile_size` is the output tile size at `target_magnification`. For 40x ROIs
    and 20x output, the source crop is 1024 px and the saved tile is 512 px.
    `label_format="auto"` handles normalized YOLO and absolute xyxy labels;
    pass "absolute_yolo" if a non-normalized file is truly YOLO center format.
    """
    roi_path = Path(roi_path)
    label_path = Path(label_path) if label_path is not None else None
    boundary_path = Path(boundary_path)

    magnification_scale = source_magnification / target_magnification
    source_tile_size = int(round(tile_size * magnification_scale))
    if source_tile_size <= 0:
        raise ValueError("source_tile_size must be positive")

    image = Image.open(roi_path).convert("RGB")
    image_size = (image.width, image.height)
    roi_polygon = read_boundary_polygon(boundary_path, image_size)
    label_boxes = read_label_boxes(label_path, image_size, label_format=label_format)

    output_root = Path(output_root) if output_root is not None else None
    safe_wsi = _safe_path_part(wsi_name or roi_path.stem)
    if output_root is not None:
        if split is None:
            raise ValueError("split is required when output_root is provided")
        tile_dir = output_root / split / safe_wsi
        tile_dir.mkdir(parents=True, exist_ok=True)
    else:
        tile_dir = None

    if include_partial_tiles:
        x_starts = range(0, image.width, source_tile_size)
        y_starts = range(0, image.height, source_tile_size)
    else:
        x_starts = range(0, max(image.width - source_tile_size + 1, 0), source_tile_size)
        y_starts = range(0, max(image.height - source_tile_size + 1, 0), source_tile_size)

    records = []
    for row_idx, y0 in enumerate(y_starts):
        for col_idx, x0 in enumerate(x_starts):
            tile_geom = box(x0, y0, x0 + source_tile_size, y0 + source_tile_size)
            roi_area = roi_polygon.intersection(tile_geom).area
            roi_fraction = roi_area / tile_geom.area if tile_geom.area else 0.0
            if roi_area == 0 or roi_fraction < min_roi_fraction:
                continue

            tile_image = _crop_and_mask_tile(image, roi_polygon, tile_geom, source_tile_size, outside_value)
            if source_tile_size != tile_size:
                tile_image = tile_image.resize((tile_size, tile_size), Image.Resampling.LANCZOS)

            tile_boxes, labels = _tile_box_records(
                label_boxes,
                tile_geom,
                source_tile_size,
                tile_size,
                object_fraction_threshold,
            )

            tile_name = f"{roi_path.stem}_r{row_idx:03d}_c{col_idx:03d}.{image_format}"
            file_path = None
            imagename = f"{safe_wsi}/{tile_name}"
            if tile_dir is not None:
                file_path = tile_dir / tile_name
                tile_image.save(file_path)
                file_path = str(file_path.resolve())

            record = {
                "dataset": "vizcarra-2023",
                "imagename": imagename,
                "tileName": tile_name,
                "filePath": file_path,
                "wsi_name": wsi_name,
                "caseId": case_id,
                "labels": labels,
                "pre_nft": labels[0],
                "inft": labels[1],
                "split": split,
                "roi_filename": roi_path.name,
                "label_filename": label_path.name if label_path is not None and label_path.exists() else None,
                "boundary_filename": boundary_path.name,
                "tile_row": row_idx,
                "tile_col": col_idx,
                "source_x": x0,
                "source_y": y0,
                "source_tile_size": source_tile_size,
                "tile_size": tile_size,
                "roi_fraction": roi_fraction,
                "boxes": tile_boxes,
            }
            if return_images:
                record["image"] = tile_image

            records.append(record)

    return pd.DataFrame(records)

## Example Tile Check

In [ ]:
example_row = train_val_split_df.iloc[0]
example_roi_path = dataset_path / "images" / example_row["filename"]
example_label_path = dataset_path / "labels" / f"{Path(example_row['filename']).stem}.txt"
example_boundary_path = dataset_path / "boundaries" / f"{Path(example_row['filename']).stem}.txt"

example_tiles_df = tile_roi(
    example_roi_path,
    example_label_path,
    example_boundary_path,
    tile_size=256,
    source_magnification=40,
    target_magnification=40,
    object_fraction_threshold=0.5,
    split=example_row["split"],
    wsi_name=example_row["wsi_name"],
    case_id=example_row["case"],
    return_images=True,
)

example_tiles_df.assign(label_tuple=example_tiles_df["labels"].map(tuple))["label_tuple"].value_counts()

In [ ]:
def show_example_tile(tile_index=0):
    if plt is None or Rectangle is None:
        raise ImportError("Install matplotlib to draw example tiles.")

    row = example_tiles_df.iloc[tile_index]
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(row["image"])
    ax.set_title(
        f"{row['tileName']} | labels={row['labels']} | ROI={row['roi_fraction']:.2f}",
        fontsize=10,
    )
    ax.axis("off")

    for bbox in row["boxes"]:
        color = CLASS_COLORS[bbox["class_id"]]
        linestyle = "-" if bbox["included"] else "--"
        rect = Rectangle(
            (bbox["x1"], bbox["y1"]),
            bbox["x2"] - bbox["x1"],
            bbox["y2"] - bbox["y1"],
            linewidth=1.5,
            edgecolor=color,
            facecolor="none",
            linestyle=linestyle,
        )
        ax.add_patch(rect)
        ax.text(
            bbox["x1"],
            max(0, bbox["y1"] - 3),
            f"{bbox['class_name']} {bbox['fraction']:.2f}",
            color="white",
            fontsize=8,
            bbox={"facecolor": color, "alpha": 0.75, "pad": 1, "edgecolor": "none"},
        )

    plt.show()


if interact is None or IntSlider is None:
    raise ImportError("Install ipywidgets to use the interactive tile viewer.")

interact(
    show_example_tile,
    tile_index=IntSlider(
        value=0,
        min=0,
        max=max(len(example_tiles_df) - 1, 0),
        step=1,
        description="tile",
        continuous_update=False,
    ),
);